In [1]:
import os, sys
from pathlib import Path

here = Path.cwd()
PROJECT_DIR = here.parent if here.name == "notebooks" else here   # the folder with make_submission.py
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))                              # so "import freight_lib" works

print("Working folder:", PROJECT_DIR)
for name in ["freight_lib.py", "make_submission.py", "score.py", "data"]:
    print(f"{name:20s}", "found" if (PROJECT_DIR / name).exists() else "MISSING")
print("CSV files:", sorted(p.name for p in (PROJECT_DIR / "data").glob("*.csv")))

import numpy, pandas, sklearn, lightgbm, matplotlib
print("Libraries OK")

Working folder: C:\Users\Nouman\Documents\freight-rate-challenge
freight_lib.py       found
make_submission.py   found
score.py             found
data                 found
CSV files: ['december-chart-inputs.csv', 'train-test.csv', 'validation-predictions-template.csv', 'validation.csv']
Libraries OK


In [2]:
import os, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
from freight_lib import (find, Cleaner, flag_corrupted, score, LookupBaseline, TARGET, FOLDS)
from make_submission import build_model

here = Path.cwd()
PROJECT_DIR = here.parent if here.name == 'notebooks' else here
DATA_DIR = PROJECT_DIR / 'data' if (PROJECT_DIR / 'data').is_dir() else Path('data')

train = pd.read_csv(find(DATA_DIR, 'train_test'), parse_dates=['date'])
val = pd.read_csv(find(DATA_DIR, 'validation'), parse_dates=['date'])
market_by_day = pd.concat([train, val]).groupby('date')['market_index'].median()

tr = Cleaner().fit(train).transform(train, market_by_day)
tr['is_corrupted'] = flag_corrupted(tr)

# --- run every fold with three models: naive, lookup, our final model
frames = []
for fold, (start, end) in FOLDS.items():
    past = tr[(tr['date'] < start) & ~tr['is_corrupted']]
    future = tr[(tr['date'] >= start) & (tr['date'] <= end)].copy()
    future['naive'] = past[TARGET].mean()
    future['lookup'] = LookupBaseline().fit(past).predict(future)
    future['final'] = build_model().fit(past).predict(future)
    future['fold'] = fold
    frames.append(future)
results = pd.concat(frames)
clean = results[~results['is_corrupted']]

# 1) Compare models, on clean rows and on all rows
for label, data in [('CLEAN rows', clean), ('ALL rows (incl. corrupted)', results)]:
    table = {m: score(data[TARGET], data[m]) for m in ['naive', 'lookup', 'final']}
    print(label); print(pd.DataFrame(table).T.round(2)); print()

# 2) Bias: is the model systematically too high or too low? (1.00 = perfect)
print('Bias by fold (sum predicted / sum actual):')
print(clean.groupby('fold').apply(lambda g: g['final'].sum() / g[TARGET].sum()).round(3))

# 3) Is it good everywhere?
print('\nWAPE by equipment:')
print(clean.groupby('equipment').apply(lambda g: score(g[TARGET], g['final'])['WAPE']).round(2))
clean['distance_band'] = pd.cut(clean['distance'], [0, 150, 500, 1200, 5000], observed=False)
print('\nWAPE by distance band:')
print(clean.groupby('distance_band', observed=False).apply(lambda g: score(g[TARGET], g['final'])['WAPE']).round(2))

# 4) Is the error small enough to use?
rel_error = (clean['final'] / clean[TARGET] - 1).abs()
print(f'\nWithin 5% of actual: {(rel_error < 0.05).mean():.1%}   Within 10%: {(rel_error < 0.10).mean():.1%}')


CLEAN rows
            MAE     RMSE   MAPE   WAPE
naive   1123.93  1359.39  79.33  47.84
lookup    83.65   116.01   3.77   3.56
final     65.46    90.84   3.00   2.79

ALL rows (incl. corrupted)
            MAE     RMSE   MAPE   WAPE
naive   1163.98  1509.20  82.18  48.90
lookup   137.20   633.99   6.19   5.76
final    119.20   629.27   5.42   5.01

Bias by fold (sum predicted / sum actual):
fold
V1    1.026
V2    0.993
V3    0.978
V4    0.995
dtype: float64

WAPE by equipment:
equipment
Dry Van    2.60
Flatbed    3.26
Reefer     2.83
dtype: float64

WAPE by distance band:
distance_band
(0, 150]        3.37
(150, 500]      3.54
(500, 1200]     3.15
(1200, 5000]    2.51
dtype: float64

Within 5% of actual: 81.5%   Within 10%: 99.2%


In [3]:
!python -m pytest -q

............                                                             [100%]
============================== warnings summary ===============================
..\..\AppData\Roaming\Python\Python312\site-packages\pytz\tzinfo.py:27
  C:\Users\Nouman\AppData\Roaming\Python\Python312\site-packages\pytz\tzinfo.py:27: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
    _epoch = datetime.utcfromtimestamp(0)

tests/test_freight_lib.py::test_flag_corrupted_catches_extreme_rates_only
tests/test_pipeline.py::test_make_submission_writes_valid_files
  C:\Users\Nouman\Documents\freight-rate-challenge\freight_lib.py:61: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the futur